In [0]:
# Cell 1: Production Incremental ETL Pipeline (Bronze to Silver Layer)
from pyspark.sql import functions as F
from pyspark.sql.window import Window
print("--- Step 1: Ingesting Raw Messy Records into our Bronze Workspace ---")
# 1. Simulate a messy Bronze landing zone with duplicates and empty status rows
raw_bronze_records = [
    ("ORD_99", "CUST_A", 100.00, None, "2026-11-01 10:00:00"),       # Has missing status
    ("ORD_100", "CUST_B", 250.00, "Pending", "2026-11-02 02:00:00"),
    ("ORD_100", "CUST_B", 250.00, "Completed", "2026-11-02 04:00:00"), # Duplicate ID with later timestamp!
    ("ORD_101", "CUST_C", 45.00, "Completed", "2026-11-02 05:00:00")
]
bronze_schema = ["order_id", "customer_id", "amount", "status", "ingest_timestamp"]
bronze_df = spark.createDataFrame(raw_bronze_records, bronze_schema)
print("--- Step 2: Cleaning Data & Removing Duplicates (The Twin Guard) ---")
# 2. Fix missing values by replacing empty statuses with 'Unknown'
clean_nulls_df = bronze_df.withColumn("status", F.coalesce(F.col("status"), F.lit("Unknown")))
# 3. Deduplicate: If an order ID appears twice, keep the most recent updated record based on timestamp
window_specification = Window.partitionBy("order_id").orderBy(F.col("ingest_timestamp").desc())
deduplicated_df = clean_nulls_df.withColumn("row_rank", F.row_number().over(window_specification)) \
                                 .filter("row_rank == 1") \
                                 .drop("row_rank")
print("--- Step 3: Saving to Silver Table with Idempotent Guarantee ---")
# 4. Initialize our formal project Silver database catalog table layer
spark.sql("CREATE DATABASE IF NOT EXISTS demo_project_db")
# 5. Execute an Upsert Merge to ensure no-matter-how-many-times-we-run-it, data stays perfect
deduplicated_df.createOrReplaceTempView("incremental_silver_source_view")
# Pre-create the target silver table schema structure if it doesn't exist
spark.sql("""
    CREATE TABLE IF NOT EXISTS demo_project_db.silver_orders (
        order_id STRING, customer_id STRING, amount DOUBLE, status STRING, ingest_timestamp STRING
    ) USING DELTA
""")
spark.sql("""
    MERGE INTO demo_project_db.silver_orders AS target
    USING incremental_silver_source_view AS source
    ON target.order_id = source.order_id
    WHEN MATCHED THEN
        UPDATE SET target.amount = source.amount, target.status = source.status, target.ingest_timestamp = source.ingest_timestamp
    WHEN NOT MATCHED THEN
        INSERT *
""")
# Display the final, pristine Silver target output grid
display(spark.sql("SELECT * FROM demo_project_db.silver_orders"))